# Foldy–Wouthuysen Derivation: The $b_\mu$ SME Coefficient

**MSc Research — Oyewo Temidayo Solomon**  
*Unified Constraint Framework for Exotic Spin-Dependent Interactions*  
University of Ibadan | Supervisor: Prof. O.E. Oyewande

---

## Overview

This notebook derives the nonrelativistic Hamiltonian arising from the CPT-odd,
Lorentz-violating $b_\mu$ coefficient of the Standard Model Extension (SME):

$$\mathcal{L}_b = -b_\mu \bar{\psi} \gamma^\mu \gamma^5 \psi$$

**Steps:**
1. Construct the Dirac-level Hamiltonian $H_b$
2. Classify even/odd operators under $\beta$
3. Apply the Foldy–Wouthuysen generator $S = -i\beta\mathcal{O}/2m$
4. Evaluate $\mathcal{O}^2$ and all BCH commutators to $O(1/m)$
5. Project onto upper components to get $H^\text{NR}$
6. Apply CPT conjugation to obtain the antiparticle Hamiltonian
7. Match to Dobrescu–Mocioiu potentials

**Reference:** Kostelecky & Lane, Phys. Rev. D 60, 116010 (1999), Eq. (4.5).


## 1. Setup — Import Dirac Algebra

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import sympy as sp
from sympy import (
    I, Matrix, symbols, sqrt, Rational, simplify, expand,
    zeros, eye, pprint, latex
)

# Import our modules (assumed in same directory or on path)
try:
    from dirac_algebra import (
        beta, alpha, gamma, gamma5, Sigma,
        sigma_munu, comm, acomm, is_even, is_odd,
        even_part, odd_part, upper, lower, I4, Z4
    )
    from pauli_matrices import sigma, I2, Z2, H_NR_bmu, cpt_conjugate
    print("Modules loaded from dirac_algebra.py and pauli_matrices.py")
except ImportError:
    print("Local modules not found — defining inline...")
    # Inline fallback definitions
    sigma_x = Matrix([[0, 1], [1, 0]])
    sigma_y = Matrix([[0, -I], [I, 0]])
    sigma_z = Matrix([[1, 0], [0, -1]])
    sigma = [sigma_x, sigma_y, sigma_z]
    I2 = eye(2); Z2 = zeros(2,2)
    I4 = eye(4); Z4 = zeros(4,4)
    beta = Matrix([[1,0,0,0],[0,1,0,0],[0,0,-1,0],[0,0,0,-1]])
    alpha = [
        Matrix([[0,0,0,1],[0,0,1,0],[0,1,0,0],[1,0,0,0]]),
        Matrix([[0,0,0,-I],[0,0,I,0],[0,-I,0,0],[I,0,0,0]]),
        Matrix([[0,0,1,0],[0,0,0,-1],[1,0,0,0],[0,-1,0,0]]),
    ]
    gamma5 = Matrix([[0,0,1,0],[0,0,0,1],[1,0,0,0],[0,1,0,0]])
    Sigma = [
        Matrix([[0,1,0,0],[1,0,0,0],[0,0,0,1],[0,0,1,0]]),
        Matrix([[0,-I,0,0],[I,0,0,0],[0,0,0,-I],[0,0,I,0]]),
        Matrix([[1,0,0,0],[0,-1,0,0],[0,0,1,0],[0,0,0,-1]]),
    ]
    def comm(A,B): return A*B - B*A
    def is_even(M): return simplify(beta*M - M*beta) == Z4
    def is_odd(M):  return simplify(beta*M + M*beta) == Z4
    def upper(M):   return M[:2,:2]

print("\nbeta ="); pprint(beta)
print("\ngamma5 ="); pprint(gamma5)


## 2. The $b_\mu$ Hamiltonian

The Dirac equation with the SME $b_\mu$ term gives the interaction Hamiltonian:

$$H_b = b_0 \gamma^0 \gamma^5 + b_i \alpha^i \gamma^5$$

Wait — we need to be careful. From $\mathcal{L}_b = -b_\mu\bar\psi\gamma^\mu\gamma^5\psi$, the
Dirac equation reads $(i\not\partial - m - b_\mu\gamma^\mu\gamma^5)\psi = 0$, so:

$$H_b = b_\mu \gamma^0 \gamma^\mu \gamma^5$$

For $\mu=0$: $\gamma^0\gamma^0\gamma^5 = \gamma^5$ (odd)  
For $\mu=i$: $\gamma^0\gamma^i\gamma^5 = \beta\alpha^i\gamma^5 = \Sigma^i\beta$ (even)

We decompose: $H_b = \mathcal{O}_b + \mathcal{E}_b$


In [ ]:
b0, b1, b2, b3 = symbols('b_0 b_1 b_2 b_3', real=True)
b_spatial = [b1, b2, b3]

# gamma^0 gamma^mu gamma^5 for each mu
g0g0g5 = beta * beta * gamma5  # = gamma5
g0gi_g5 = [beta * alpha[i] * gamma5 for i in range(3)]

print("gamma^0 * gamma^0 * gamma^5 (mu=0 contribution, should be gamma5):")
pprint(g0g0g5)
print("\nIs it odd (off-diagonal)?", is_odd(g0g0g5))

print("\ngamma^0 * gamma^i * gamma^5 for i=1 (mu=1 contribution):")
pprint(g0gi_g5[0])
print("Is it even (block-diagonal)?", is_even(g0gi_g5[0]))

# Verify: beta * alpha^i * gamma^5 = Sigma^i * beta
print("\nVerify beta*alpha[0]*gamma5 = Sigma[0]*beta:")
lhs = beta * alpha[0] * gamma5
rhs = Sigma[0] * beta
print("LHS - RHS =", simplify(lhs - rhs))


In [ ]:
# Full Hamiltonian contribution from b_mu
H_b_odd  = b0 * gamma5                              # O_b : odd part
H_b_even = sum(b_spatial[i] * Sigma[i] * beta       # E_b : even part
               for i in range(3))

H_b = H_b_odd + H_b_even

print("Odd part of H_b (O_b = b0 * gamma5):")
pprint(H_b_odd)
print("\nIs O_b odd?", is_odd(H_b_odd))

print("\nEven part of H_b (E_b = b_i Sigma^i beta):")
pprint(H_b_even)
print("Is E_b even?", is_even(H_b_even))


## 3. Full Hamiltonian and Even/Odd Decomposition

The full Hamiltonian (including free Dirac and EM field terms) is:

$$H = \underbrace{\beta m}_{\mathcal{E}_0} + \underbrace{\boldsymbol{\alpha}\cdot\mathbf{p}}_{\mathcal{O}_0} + \underbrace{e\Phi}_{\mathcal{E}_\Phi} + \underbrace{\mathbf{b}\cdot\boldsymbol{\Sigma}\beta}_{\mathcal{E}_b} + \underbrace{b_0\gamma^5}_{\mathcal{O}_b}$$

The total odd operator driving the FW generator is $\mathcal{O} = \boldsymbol{\alpha}\cdot\mathbf{p} + b_0\gamma^5$.


In [ ]:
def _msum(terms):
    """Matrix-safe sum."""
    from sympy import zeros
    result = None
    for t in terms:
        result = t if result is None else result + t
    return result if result is not None else zeros(4,4)

# Symbolic momentum components
p1, p2, p3 = symbols('p_1 p_2 p_3', real=True)
m_sym = symbols('m', positive=True)
p_vec = [p1, p2, p3]

# Construct alpha.p  (odd)
alpha_dot_p = _msum([p_vec[i] * alpha[i] for i in range(3)])

print("alpha . p:")
pprint(alpha_dot_p)
print("Is alpha.p odd?", is_odd(alpha_dot_p))

# Total odd operator
O_total = alpha_dot_p + b0 * gamma5
print("\nTotal odd operator O = alpha.p + b0*gamma5:")
pprint(O_total)


## 4. The FW Generator

We choose $S = -\frac{i\beta\mathcal{O}}{2m}$ to cancel $\mathcal{O}$ at leading order.

**Verification:** $i[S, \beta m] = -\mathcal{O}$ must hold.


In [ ]:
# FW generator S = -i beta O / 2m
S_bmu = -I * beta * O_total / (2 * m_sym)

print("FW generator S (symbolic, showing structure):")
pprint(S_bmu)

# Verify: i[S, beta*m] = -O_total
commutator = I * comm(S_bmu, beta * m_sym)
residual = simplify(commutator + O_total)
print("\nVerification: i[S, beta*m] + O_total =")
pprint(residual)
print("=> Zero?", residual == Z4)


## 5. Computing $\mathcal{O}^2$ to First Order in $b_\mu$

$$\mathcal{O}^2 = (\boldsymbol{\alpha}\cdot\mathbf{p})^2 + 2b_0\,\boldsymbol{\Sigma}\cdot\mathbf{p} + O(b^2)$$

We use $(\boldsymbol{\alpha}\cdot\mathbf{p})^2 = \mathbf{p}^2$ and $[\alpha^i, \gamma^5] = 0$.


In [ ]:
# O^2 symbolic
O_sq = simplify(O_total * O_total)
print("O^2 (full, symbolic):")
pprint(O_sq)


In [ ]:
def _msum(terms):
    """Matrix-safe sum."""
    from sympy import zeros
    result = None
    for t in terms:
        result = t if result is None else result + t
    return result if result is not None else zeros(4,4)

# Verify (alpha.p)^2 = p^2 * I4
alpha_p_sq = simplify(alpha_dot_p * alpha_dot_p)
p_sq = p1**2 + p2**2 + p3**2
print("(alpha.p)^2 - p^2*I4 =")
pprint(simplify(alpha_p_sq - p_sq * I4))

# Cross term: (alpha.p)(b0*gamma5) + (b0*gamma5)(alpha.p)
cross = simplify(alpha_dot_p * (b0*gamma5) + (b0*gamma5) * alpha_dot_p)
print("\nCross term (alpha.p)(b0*g5) + (b0*g5)(alpha.p):")
pprint(cross)

# Should equal 2*b0 * Sigma.p
Sigma_dot_p = _msum([p_vec[i] * Sigma[i] for i in range(3)])
expected_cross = 2 * b0 * Sigma_dot_p
print("\n2*b0 * Sigma.p:")
pprint(expected_cross)
print("\nMatch?", simplify(cross - expected_cross) == Z4)


## 6. Nonrelativistic Hamiltonian — Upper Components

After the FW transformation, projecting onto upper (positive-energy) components via $\beta \to +1$:

$$H^\text{NR}_{b_\mu} = \frac{\mathbf{p}^2}{2m} + \mathbf{b}\cdot\boldsymbol{\sigma} + \frac{b_0}{m}\boldsymbol{\sigma}\cdot\mathbf{p} + O(1/m^2)$$

The **leading spin-dependent term** (dropping kinetic energy and rest mass):

$$\boxed{H^\text{NR}_{b} = -\mathbf{b}\cdot\boldsymbol{\sigma}}$$


In [ ]:
# Upper 2x2 block of O^2
O_sq_upper = upper(O_sq)
print("Upper 2x2 block of O^2:")
pprint(O_sq_upper)

# H^NR from FW: beta*O^2/(2m) upper block = O_sq_upper / (2m)
H_kinetic = (p1**2 + p2**2 + p3**2) / (2*m_sym) * I2
H_b_upper_from_O2 = simplify(O_sq_upper / (2 * m_sym) - H_kinetic)
print("\nSpin-dependent part from O^2/(2m) [upper]:")
pprint(H_b_upper_from_O2)

# Even part E_b contributes directly: upper block of b_i Sigma^i beta
# (beta -> +1 for upper)
E_b_upper = upper(H_b_even)
print("\nEven part E_b [upper block, beta=+1]:")
pprint(E_b_upper)

# Total spin-dependent H^NR (leading order, dropping p^2/2m)
H_NR_spin = simplify(H_b_upper_from_O2 + E_b_upper)
print("\nTotal spin-dependent H^NR_b (leading order):")
pprint(H_NR_spin)


In [ ]:
# Use pauli_matrices.py convenience function
try:
    H_check, desc = H_NR_bmu([b1, b2, b3])
    print("From pauli_matrices.H_NR_bmu:")
    print(desc)
    pprint(H_check)

    # Eigenvalues = spin splitting
    eigs = H_check.eigenvals()
    print("\nEigenvalues (energy splitting in b-field):")
    for e, mult in eigs.items():
        print(f"  E = {e}  (multiplicity {mult})")
    print("\nEnergy splitting = 2|b| =", simplify(2*sqrt(b1**2+b2**2+b3**2)))
except Exception as ex:
    print("pauli_matrices not available:", ex)


## 7. CPT Conjugation — Antiparticle Sector

For CPT-odd coefficients, the lower (antiparticle) components give:

$$H^\text{NR}_{b,\,\bar{f}} = +\mathbf{b}\cdot\boldsymbol{\sigma}$$

The sign is **opposite** to the particle case. This is the direct theoretical prediction of CPT violation detectable via $A_\alpha$.


In [ ]:
# Lower 2x2 block of E_b  (beta -> -1 for antiparticle)
E_b_lower = lower(H_b_even)
print("Even part E_b [lower block, beta=-1]:")
pprint(E_b_lower)

# Note: the sign flips because Sigma^i beta -> Sigma^i*(-1) in lower block
# So H^NR_{b,antiparticle} = +b.sigma

print("\n=> For antimatter: H^NR_b = +b.sigma (CPT-odd sign flip)")
print("   For matter:      H^NR_b = -b.sigma")
print("\nCPT test: measure spin precession frequency in both sectors.")
print("If they differ => b_mu != 0 => CPT violated.")


## 8. Connection to Dobrescu–Mocioiu Potentials

The uniform background $\mathbf{b}\cdot\boldsymbol{\sigma}$ corresponds to the **long-range limit**
($\lambda \to \infty$) of the monopole-dipole potential $V_2$:

$$V_2 = \frac{g_s g_p}{8\pi m_2}\,\boldsymbol{\sigma}_2\cdot\hat{\mathbf{r}}
\left(\frac{1}{\lambda r} + \frac{1}{r^2}\right)e^{-r/\lambda}$$

The identification for a macroscopic source (summing over source particles) is:

$$b_i \leftrightarrow \frac{g_s g_p}{8\pi m}\,\langle\hat{r}_i\rangle_\text{source}$$

For your $g_Ag_A$ pairs (V2 sector), the Fadeev2022 antimatter constraints probe
exactly this coupling in the positron/antiproton sector.

The asymmetry parameter:

$$A_\alpha = \frac{g^{(f)}_\alpha - g^{(\bar{f})}_\alpha}{g^{(f)}_\alpha + g^{(\bar{f})}_\alpha}$$

In the pure $b_\mu$ scenario, theory predicts $|A_\alpha| = 1$ because the coupling
constants extracted from matter and antimatter experiments have opposite signs —
consistent with your measured values of $|A_\alpha| \sim 1.000$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Illustrate the spin energy splitting as a function of b-field direction
theta = np.linspace(0, 2*np.pi, 300)
b_mag = 1.0  # in units of the coupling

# b = b_mag * (sin(theta), 0, cos(theta))  — rotation in xz-plane
E_plus  =  b_mag * np.ones_like(theta)   # +|b|  (spin antiparallel to b)
E_minus = -b_mag * np.ones_like(theta)   # -|b|  (spin parallel to b)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: energy splitting
ax = axes[0]
ax.axhline( b_mag, color='steelblue', lw=2, label=r'$E_+ = +|\mathbf{b}|$ (matter)')
ax.axhline(-b_mag, color='steelblue', lw=2, ls='--', label=r'$E_- = -|\mathbf{b}|$ (matter)')
ax.axhline( b_mag, color='tomato',    lw=2, ls=':', alpha=0.7, label=r'$E_- = -|\mathbf{b}|$ (antimatter, CPT-odd flip)')
ax.axhline(-b_mag, color='tomato',    lw=2, ls='-.', alpha=0.7, label=r'$E_+ = +|\mathbf{b}|$ (antimatter, CPT-odd flip)')
ax.set_ylim(-2, 2)
ax.set_xlabel(r'(uniform — independent of direction)', fontsize=11)
ax.set_ylabel(r'Energy / $|\mathbf{b}|$', fontsize=11)
ax.set_title(r'Spin energy levels: $H^\mathrm{NR}_b = -\mathbf{b}\cdotoldsymbol{\sigma}$', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Right: CPT asymmetry A_alpha as function of ratio g_fbar/g_f
r = np.linspace(-2, 2, 400)
A = (1 - r) / (1 + r + 1e-10)  # avoid division by zero
A = np.clip(A, -3, 3)

ax2 = axes[1]
ax2.plot(r, A, 'k-', lw=2)
ax2.axhline(1,  color='tomato',    ls='--', lw=1.5, label=r'$|A_lpha|=1$ (pure CPT-odd)')
ax2.axhline(-1, color='tomato',    ls='--', lw=1.5)
ax2.axhline(0,  color='steelblue', ls=':', lw=1.5, label=r'$A_lpha=0$ (CPT symmetric)')
ax2.axvline(-1, color='gray', ls=':', lw=1)
ax2.scatter([-1], [float('nan')], s=0)  # just for spacing
ax2.set_xlabel(r'$g^{(ar{f})}_lpha \;/\; g^{(f)}_lpha$', fontsize=12)
ax2.set_ylabel(r'$A_lpha$', fontsize=12)
ax2.set_title(r'Asymmetry parameter vs coupling ratio', fontsize=12)
ax2.set_ylim(-3, 3)
ax2.set_xlim(-2, 2)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('FW_bmu_asymmetry.pdf', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: FW_bmu_asymmetry.pdf")


## Summary

| Quantity | Result |
|---|---|
| SME term | $\mathcal{L}_b = -b_\mu\bar\psi\gamma^\mu\gamma^5\psi$ |
| CPT property | **Odd** — sign flips under CPT |
| Odd operator | $\mathcal{O}_b = b_0\gamma^5$ |
| Even operator | $\mathcal{E}_b = \mathbf{b}\cdot\boldsymbol{\Sigma}\beta$ |
| FW generator | $S = -i\beta(\boldsymbol{\alpha}\cdot\mathbf{p} + b_0\gamma^5)/(2m)$ |
| $H^\text{NR}$ (matter) | $-\mathbf{b}\cdot\boldsymbol{\sigma}$ |
| $H^\text{NR}$ (antimatter) | $+\mathbf{b}\cdot\boldsymbol{\sigma}$ |
| DM potential | $V_2$ (long-range limit) |
| Expected $|A_\alpha|$ | 1 (exact CPT-odd prediction) |
| Observed $|A_\alpha|$ (SPINDEP) | $\geq 0.954$ for all $g_Ag_A$ pairs |
